In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!pip install torch transformers datasets sentencepiece indic-nlp-library
!pip install bert-score sacrebleu wandb

  Using cached bert_score-0.3.13-py3-none-any.whl.metadata (15 kB)
  Using cached sacrebleu-2.5.1-py3-none-any.whl.metadata (51 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 6.9 MB/s eta 0:00:00


In [3]:
!git clone https://github.com/MIRAH-Official/Empathetic-Chatbot-ASEM.git
!git clone https://github.com/facebookresearch/XLM.git
!git clone https://github.com/az-raei/DSL501_ML_Project.git

Cloning into 'Empathetic-Chatbot-ASEM'...
remote: Enumerating objects: 156, done.
remote: Counting objects: 100% (156/156), done.
remote: Compressing objects: 100% (108/108), done.
remote: Total 156 (delta 61), reused 136 (delta 41), pack-reused 0 (from 0)
Receiving objects: 100% (156/156), 30.99 MiB | 15.05 MiB/s, done.
Resolving deltas: 100% (61/61), done.
Updating files: 100% (41/41), done.
Cloning into 'XLM'...
remote: Enumerating objects: 358, done.
remote: Counting objects: 100% (218/218), done.
remote: Compressing objects: 100% (36/36), done.
remote: Total 358 (delta 184), reused 182 (delta 182), pack-reused 140 (from 1)
Receiving objects: 100% (358/358), 182.91 KiB | 6.31 MiB/s, done.
Resolving deltas: 100% (218/218), done.
Cloning into 'DSL501_ML_Project'...
remote: Enumerating objects: 6, done.
remote: Counting objects: 100% (6/6), done.
remote: Compressing objects: 100% (5/5), done.
remote: Total 6 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100%

In [5]:
!pip install transformers datasets sentencepiece indic-nlp-library
!pip install openai


In [6]:
!mkdir -p /content/drive/MyDrive/hygieia/{data,models,notebooks}


In [8]:
!pip install transformers indic-nlp-library sentencepiece


In [9]:
from transformers import AutoTokenizer, AutoModelForMaskedLM, pipeline

tokenizer = AutoTokenizer.from_pretrained("ai4bharat/indic-bert")
model = AutoModelForMaskedLM.from_pretrained("ai4bharat/indic-bert")

fill_mask = pipeline("fill-mask", model=model, tokenizer=tokenizer)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/507 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/5.65M [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/135M [00:00<?, ?B/s]

Device set to use cuda:0


In [10]:
def generate_empathy_turn(prompt):
    outputs = fill_mask(prompt)
    return outputs[0]["sequence"]


In [11]:
text = "क्लाइंट: मुझे बहुत चिंता हो रही है। \nकाउंसलर: [MASK] शांत रहिए, आप अकेले नहीं हैं।"
print(generate_empathy_turn(text))


model.safetensors:   0%|          | 0.00/135M [00:00<?, ?B/s]

कलइट: मझ बहत चत ह रह ह। कउसलर: आप शत रहए, आप अकल नह ह।


In [15]:
from datasets import Dataset

client_prompts = [
    "मुझे हाल ही में नींद नहीं आ रही है।",
    "मैं अपने परीक्षा परिणाम को लेकर बहुत तनाव में हूँ।",
    "परिवार के झगड़ों से मन बहुत भारी है।",
    "मुझे लगता है कि कोई मेरी बात नहीं समझता।",
    "मेरे दोस्त अब पहले जैसे नहीं रहे।",
    "काम का दबाव बढ़ गया है और मैं थक गया हूँ।"
]

counselor_templates = [
    "काउंसलर: [MASK] मैं आपकी भावना समझ सकता हूँ।",
    "काउंसलर: [MASK] चलिए इस पर बात करते हैं।",
    "काउंसलर: [MASK] यह सामान्य है, थोड़ा समय खुद को दीजिए।",
    "काउंसलर: [MASK] आपकी बात सुनकर अच्छा लगा कि आप खुलकर साझा कर रहे हैं।"
]

data = []
for client in client_prompts:
    for template in counselor_templates:
        data.append({
            "client": client,
            "prompt": template
        })

dataset = Dataset.from_list(data)
dataset

Dataset({
    features: ['client', 'prompt'],
    num_rows: 24
})

In [16]:
def batch_fill_mask(batch):
    filled = fill_mask(batch["prompt"], top_k=1)
    results = []
    for i, outputs in enumerate(filled):
        if isinstance(outputs, list):  # pipeline returns list per sample
            results.append(outputs[0]["sequence"])
        else:
            results.append(outputs["sequence"])
    batch["filled_prompt"] = results
    return batch

batched_dataset = dataset.map(batch_fill_mask, batched=True, batch_size=8)


Map:   0%|          | 0/24 [00:00<?, ? examples/s]

In [17]:
def combine_dialogue(batch):
    batch["dialogue"] = [f"क्लाइंट: {c}\n{r}" for c, r in zip(batch["client"], batch["filled_prompt"])]
    return batch

batched_dataset = batched_dataset.map(combine_dialogue)


Map:   0%|          | 0/24 [00:00<?, ? examples/s]

In [18]:
output_path = "/content/drive/MyDrive/hygieia/data/hindi_indicbert_batched.json"
batched_dataset.to_json(output_path, force_ascii=False)
print(f"✅ Saved {len(batched_dataset)} dialogues to {output_path}")

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

✅ Saved 24 dialogues to /content/drive/MyDrive/hygieia/data/hindi_indicbert_batched.json
